# Project CV: local environment and data check

This notebook validates the local WSL2/ROS 2 runtime. It does not install packages, download data, or write into the source data directories.

Select the `Project CV (ROS 2 Humble)` kernel before running it.

In [1]:
import os
import platform
import sys

import cv2
import gtsam
import gtsam_unstable
import numpy as np
import rosbag2_py

assert platform.system() == 'Linux', platform.system()
assert sys.version_info[:2] == (3, 10), platform.python_version()
assert os.environ.get('ROS_DISTRO') == 'humble', os.environ.get('ROS_DISTRO')
assert hasattr(gtsam_unstable, 'IncrementalFixedLagSmoother')

print('Python:', platform.python_version())
print('ROS_DISTRO:', os.environ['ROS_DISTRO'])
print('NumPy:', np.__version__)
print('OpenCV:', cv2.__version__)
print('GTSAM fixed-lag: OK')

Python: 3.10.12
ROS_DISTRO: humble
NumPy: 1.26.4
OpenCV: 4.5.4
GTSAM fixed-lag: OK


In [2]:
from pathlib import Path

required_environment = [
    'PROJECT_CV_SOURCE',
    'PROJECT_CV_RUNTIME',
    'PROJECT_CV_RAW_BAG',
    'PROJECT_CV_DERIVED_BAG',
    'PROJECT_CV_CALIBRATION',
    'PROJECT_CV_ARTIFACTS',
]
missing_environment = [name for name in required_environment if not os.environ.get(name)]
assert not missing_environment, f'Missing environment variables: {missing_environment}'

raw_bag = Path(os.environ['PROJECT_CV_RAW_BAG'])
derived_bag = Path(os.environ['PROJECT_CV_DERIVED_BAG'])
calibration = Path(os.environ['PROJECT_CV_CALIBRATION'])
artifacts = Path(os.environ['PROJECT_CV_ARTIFACTS'])

required_paths = [
    raw_bag / 'metadata.yaml',
    raw_bag / 'K2R00005_20260607_194949_0.mcap',
    derived_bag / 'metadata.yaml',
    derived_bag / 'K2R00005_20260607_194949_with_velocity.mcap',
    calibration / 'thermal_camera.yml',
    calibration / 'cam_to_imu_rot_mtrx.yml',
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
assert not missing_paths, f'Missing runtime paths: {missing_paths}'
assert artifacts.is_dir()

print('Raw bag:', raw_bag)
print('Derived bag:', derived_bag)
print('Calibration:', calibration)
print('Writable artifacts:', artifacts)

Raw bag: /home/fedor/project_cv_runtime/bags/raw_k2r
Derived bag: /home/fedor/project_cv_runtime/bags/derived_velocity
Calibration: /home/fedor/project_cv_runtime/calibration
Writable artifacts: /home/fedor/project_cv_runtime/artifacts


In [3]:
import pandas as pd
from IPython.display import display
from rosbag2_py import ConverterOptions, SequentialReader, StorageOptions

reader = SequentialReader()
reader.open(
    StorageOptions(uri=str(derived_bag), storage_id='mcap'),
    ConverterOptions(
        input_serialization_format='cdr',
        output_serialization_format='cdr',
    ),
)
topic_rows = [
    {'topic': item.name, 'type': item.type}
    for item in reader.get_all_topics_and_types()
]
topic_table = pd.DataFrame(topic_rows).sort_values('topic', ignore_index=True)
assert '/vision/velocity_frd' in set(topic_table['topic'])
assert '/device/starlink/raw' in set(topic_table['topic'])
display(topic_table)

first_topic, first_payload, first_timestamp = reader.read_next()
print('First message:', first_topic, '@', first_timestamp)
print('Derived MCAP read test: OK')

,topic,type
0,/camera/image_raw,sensor_msgs/msg/Image
1,/camera/image_raw/compressed,sensor_msgs/msg/CompressedImage
2,/device/gps/raw,sensor_msgs/msg/NavSatFix
3,/device/starlink/raw,sensor_msgs/msg/NavSatFix
4,/device/xps/fix,sensor_msgs/msg/NavSatFix
5,/device/xps_sdl,std_msgs/msg/String
6,/imu/data_raw,sensor_msgs/msg/Imu
7,/imu/euler,geometry_msgs/msg/Vector3Stamped
8,/imu/pure_baro_alt,std_msgs/msg/Float64
9,/mavros/global_position/raw/fix,sensor_msgs/msg/NavSatFix


First message: /device/gps/raw @ 1780850989877460379
Derived MCAP read test: OK


## Next stage

The original Colab notebook remains a read-only reference. After this check passes, continue in `01_sparse_gps_fusion_local.ipynb`.